# nb_03_ingest_to_index — Doc Intelligence → chunk → embed → AI Search

**Pipeline 2.** Works the `file_metadata` status queue: claims changed/new files, enforces ACLs,
extracts with Document Intelligence, chunks (with page numbers), embeds via Azure OpenAI, and
pushes to Azure AI Search — stamping `allowed_groups` for security trimming. Handles deletions,
re-ingest cleanup, retries/dead-letter, and logging.

> **Do not run this concurrently with `nb_01`.** Both write the same `file_metadata` Delta table;
> run them in sequence (`nb_01` → `nb_03`). The drain loop is fully restartable. See PRODUCT_SPEC.md §8.

See `PRODUCT_SPEC.md` sections 5, 7.3, 10, 11.

Every external service is called over **REST with `requests`** (always present in Fabric) — there
is **no `%pip install`** and no `azure-*`/`openai` SDK, because inline pip is rejected by the
Fabric job runtime and those SDKs aren't in the base image.

## Required permissions (identity running this notebook)
Auth is **hybrid** (a Fabric constraint):

| Resource | Auth in Fabric | Requirement |
| --- | --- | --- |
| Document Intelligence | Entra `Bearer` token via `notebookutils` | running user needs **Cognitive Services User** |
| Azure OpenAI | Entra `Bearer` token via `notebookutils` | running user needs **Cognitive Services OpenAI User** |
| Azure AI Search | **Admin API key** from Key Vault (`kv_name`/`search_key_secret`) | user needs KV **secret get** |

Cognitive Services enforce `disableLocalAuth=true` (keyless only); AI Search is not under that
policy so it uses a key held in Key Vault. The Fabric lakehouse with the S3 shortcut must be
attached (files are read from `/lakehouse/default/Files/...`).


## Config


In [ ]:
import uuid, time, json, hashlib, io, os, random, requests, traceback, threading
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable

cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
RUN_ID = str(uuid.uuid4())
ACL_BYPASS = cfg.get('acl_bypass_enabled', 'false').lower() == 'true'
SUPPORTED = set(cfg.get('supported_extensions', '').split(','))
MAX_RETRIES = int(cfg.get('max_retries', '3'))
CHUNK_SIZE = int(cfg.get('chunk_size', '1000'))
STRATEGY = cfg.get('chunk_strategy_version', 'v1')
EMB_DEPLOY = cfg['aoai_embedding_deployment']
MAX_CONC = int(cfg.get('max_concurrency', '8'))
BACKFILL = cfg.get('backfill_mode', 'false').lower() == 'true'
BATCH = int(cfg.get('backfill_batch_size', '500')) if BACKFILL else int(cfg.get('batch_size', '100'))
MAX_BATCHES = int(cfg.get('max_batches_per_run', '0'))     # 0 = drain the whole queue
RUN_BUDGET_S = int(cfg.get('run_time_budget_min', '0')) * 60  # 0 = no wall-clock cap
print('run_id:', RUN_ID, '| acl_bypass:', ACL_BYPASS)


## S3 SigV4 REST helpers (used only when `source_mode=s3_direct`)
Pure `requests` + stdlib — no boto3, no `%pip`. Mirrors `scripts/s3_rest.py`. In `s3_direct`
mode files are read straight from S3; in `s3_shortcut` mode they are read from the OneLake
local mount (below).


In [ ]:
import hashlib, hmac, os, time
import datetime as _dt
import urllib.parse as _url
import xml.etree.ElementTree as _ET
import requests

_EMPTY_SHA256 = hashlib.sha256(b"").hexdigest()

def _s3_sign(key, msg):
    return hmac.new(key, msg.encode("utf-8"), hashlib.sha256).digest()

def _s3_signing_key(secret, datestamp, region, service):
    k_date = _s3_sign(("AWS4" + secret).encode("utf-8"), datestamp)
    k_region = hmac.new(k_date, region.encode("utf-8"), hashlib.sha256).digest()
    k_service = hmac.new(k_region, service.encode("utf-8"), hashlib.sha256).digest()
    return hmac.new(k_service, b"aws4_request", hashlib.sha256).digest()

def _s3_canonical_query(params):
    if not params:
        return ""
    items = []
    for k in sorted(params):
        v = "" if params[k] is None else str(params[k])
        items.append(_url.quote(str(k), safe="") + "=" + _url.quote(v, safe=""))
    return "&".join(items)

def _s3_endpoint_parts(endpoint_url, bucket, key, addressing):
    p = _url.urlparse(endpoint_url)
    scheme = p.scheme or "https"
    ep_host = p.netloc
    enc_key = _url.quote(key, safe="/")
    if addressing == "virtual":
        host = bucket + "." + ep_host
        canonical_uri = "/" + enc_key
    else:
        host = ep_host
        canonical_uri = "/" + bucket + (("/" + enc_key) if key else "")
    return host, canonical_uri, scheme + "://" + host + canonical_uri

def _s3_signed_headers(method, host, canonical_uri, params, region, ak, sk, service="s3"):
    now = _dt.datetime.now(_dt.timezone.utc)
    amzdate = now.strftime("%Y%m%dT%H%M%SZ")
    datestamp = now.strftime("%Y%m%d")
    canonical_qs = _s3_canonical_query(params)
    canonical_headers = ("host:" + host + "\n"
                         "x-amz-content-sha256:" + _EMPTY_SHA256 + "\n"
                         "x-amz-date:" + amzdate + "\n")
    signed_headers = "host;x-amz-content-sha256;x-amz-date"
    canonical_request = "\n".join([method, canonical_uri, canonical_qs,
                                   canonical_headers, signed_headers, _EMPTY_SHA256])
    scope = datestamp + "/" + region + "/" + service + "/aws4_request"
    string_to_sign = "\n".join(["AWS4-HMAC-SHA256", amzdate, scope,
                                hashlib.sha256(canonical_request.encode("utf-8")).hexdigest()])
    signature = hmac.new(_s3_signing_key(sk, datestamp, region, service),
                         string_to_sign.encode("utf-8"), hashlib.sha256).hexdigest()
    authorization = ("AWS4-HMAC-SHA256 Credential=" + ak + "/" + scope +
                     ", SignedHeaders=" + signed_headers + ", Signature=" + signature)
    return {"Authorization": authorization, "x-amz-date": amzdate,
            "x-amz-content-sha256": _EMPTY_SHA256}

def _s3_signed_get(endpoint_url, bucket, key, region, ak, sk, params=None,
                   addressing="path", verify=True, stream=False, timeout=(10, 300)):
    host, canonical_uri, url = _s3_endpoint_parts(endpoint_url, bucket, key, addressing)
    headers = _s3_signed_headers("GET", host, canonical_uri, params, region, ak, sk)
    resp = requests.get(url, headers=headers, params=params, verify=verify,
                        stream=stream, timeout=timeout)
    if 300 <= resp.status_code < 400:
        raise requests.HTTPError("S3 " + str(resp.status_code) +
                                 " redirect (wrong region/endpoint?): " + resp.text[:300])
    return resp

def s3_list_objects(endpoint_url, bucket, region, ak, sk, prefix="",
                    addressing="path", verify=True, timeout=(10, 60)):
    """ListObjectsV2 across continuation tokens -> [{key,size,last_modified,etag}]."""
    ns = "{http://s3.amazonaws.com/doc/2006-03-01/}"
    out, token = [], None
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if token:
            params["continuation-token"] = token
        r = _s3_signed_get(endpoint_url, bucket, "", region, ak, sk,
                           params=params, addressing=addressing, verify=verify, timeout=timeout)
        r.raise_for_status()
        root = _ET.fromstring(r.content)
        for c in root.findall(ns + "Contents"):
            k = c.findtext(ns + "Key")
            if not k or k.endswith("/"):
                continue
            out.append({"key": k,
                        "size": int(c.findtext(ns + "Size") or 0),
                        "last_modified": c.findtext(ns + "LastModified"),
                        "etag": (c.findtext(ns + "ETag") or "").strip('"')})
        truncated = (root.findtext(ns + "IsTruncated") or "false").lower() == "true"
        token = root.findtext(ns + "NextContinuationToken")
        if not truncated or not token:
            break
    return out

def s3_get_bytes(endpoint_url, bucket, key, region, ak, sk,
                 addressing="path", verify=True, timeout=(10, 300)):
    r = _s3_signed_get(endpoint_url, bucket, key, region, ak, sk,
                       addressing=addressing, verify=verify, timeout=timeout)
    r.raise_for_status()
    return r.content


## Source-layer config
Resolves `source_mode` and (for `s3_direct`) the S3 endpoint/bucket/region + credentials from
Key Vault. File identity everywhere is the **src_key** (path relative to the bucket root).


In [ ]:
# Source-layer config (mode-independent identity = src_key = object path relative to bucket root).
SOURCE_MODE   = cfg.get("source_mode", "s3_shortcut").strip().lower()   # s3_shortcut | s3_direct
SHORTCUT_ROOT = cfg.get("shortcut_root", "Files/s3_mmx_bucket").rstrip("/")
S3_ENDPOINT   = cfg.get("s3_endpoint_url", "").rstrip("/")
S3_BUCKET     = cfg.get("s3_bucket", "")
S3_PREFIX     = cfg.get("s3_prefix", "")
S3_REGION     = cfg.get("s3_region", "us-east-1")
S3_ADDRESSING = cfg.get("s3_addressing", "path").strip().lower()        # path | virtual
S3_VERIFY_TLS = cfg.get("s3_verify_tls", "true").strip().lower() != "false"

_S3_AK = _S3_SK = None
if SOURCE_MODE == "s3_direct":
    import notebookutils
    _VAULT = "https://" + cfg["kv_name"] + ".vault.azure.net/"
    _S3_AK = notebookutils.credentials.getSecret(_VAULT, cfg.get("s3_access_key_secret", "s3-access-key"))
    _S3_SK = notebookutils.credentials.getSecret(_VAULT, cfg.get("s3_secret_key_secret", "s3-secret-key"))
    if not (S3_ENDPOINT and S3_BUCKET):
        raise RuntimeError("source_mode=s3_direct requires s3_endpoint_url and s3_bucket in config")
    print("source: s3_direct ->", S3_ENDPOINT, "bucket=" + S3_BUCKET,
          "prefix=" + repr(S3_PREFIX), "region=" + S3_REGION, "addr=" + S3_ADDRESSING)
else:
    print("source: s3_shortcut ->", SHORTCUT_ROOT)


## Auth (Fabric-native tokens + Search key)
Fabric cannot use `DefaultAzureCredential`. **Document Intelligence** and **Azure OpenAI** accept
an Entra token from `notebookutils.credentials.getToken('https://cognitiveservices.azure.com')`
(the running user needs the Cognitive Services roles above), sent as `Authorization: Bearer`.
**AI Search** uses an **admin API key** from Key Vault (its token audience isn't issuable in
Fabric), sent in the `api-key` header. `cog_token()` is called fresh per request so long runs
don't hit token expiry.


In [ ]:
import notebookutils

COG_SCOPE = 'https://cognitiveservices.azure.com'
def cog_token():
    # Fabric returns a cached, still-valid Entra token for the running user.
    return notebookutils.credentials.getToken(COG_SCOPE)

# AI Search admin key from Key Vault (Search token audience isn't issuable in Fabric).
VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
SEARCH_KEY = notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret'])
SEARCH_ENDPOINT = cfg['search_endpoint'].rstrip('/')
INDEX_NAME = cfg['search_index_name']
SEARCH_API = '2024-07-01'
AOAI_ENDPOINT = cfg['aoai_endpoint'].rstrip('/')
AOAI_API = '2024-06-01'
print('search key len =', len(SEARCH_KEY))


## Resilient HTTP (timeouts + retry/backoff)
Every external call goes through `http()`, which enforces connect/read **timeouts** (so a stalled
socket can never hang the run) and **retries** transient failures — `429` and `5xx` — with
exponential backoff that honors the `Retry-After` header. This is what lets the pipeline stay
healthy against Document Intelligence / Azure OpenAI **rate limits** at scale.


In [ ]:
RETRY_STATUS = {429, 500, 502, 503, 504}

# --- Throttle / retry visibility -------------------------------------------------------------
# Rate-limiting used to be invisible: http() silently retried 429/5xx and you only saw the
# effect as reduced run_progress throughput. Now every transient retry AND any attempt-
# exhaustion is recorded into a thread-safe buffer (safe under the ThreadPoolExecutor). The
# drain loop flushes the buffer to the `throttle_log` Delta table once per batch, so throttling
# is a first-class, queryable signal (which endpoint, how often, how long we waited).
_THROTTLE_LOCK = threading.Lock()
_THROTTLE_BUF = []
def _host(url):
    try:
        return url.split('://', 1)[1].split('/', 1)[0]
    except Exception:
        return str(url)[:80]
def _record_throttle(kind, url, method, status, attempt, wait_s):
    ev = (datetime.now(timezone.utc), _host(url), method, int(status or 0),
          int(attempt), float(wait_s or 0.0), kind)
    with _THROTTLE_LOCK:
        _THROTTLE_BUF.append(ev)

def http(method, url, *, headers=None, data=None, json_body=None,
         timeout=(10, 120), max_attempts=6):
    """Timed, retrying HTTP. Returns the final Response (caller does raise_for_status).
    Records every transient retry + any exhaustion via _record_throttle for observability."""
    last = None
    for attempt in range(max_attempts):
        try:
            last = requests.request(method, url, headers=headers, data=data,
                                    json=json_body, timeout=timeout)
        except requests.exceptions.RequestException as e:
            # network/timeout error: back off and retry
            if attempt == max_attempts - 1:
                _record_throttle('network_exhausted', url, method, 0, attempt + 1, 0)
                raise
            wait = min(2 ** attempt, 30) + random.uniform(0, 1)
            _record_throttle('network_retry', url, method, 0, attempt + 1, wait)
            time.sleep(wait); continue
        if last.status_code in RETRY_STATUS and attempt < max_attempts - 1:
            ra = last.headers.get('Retry-After')
            wait = (float(ra) if ra else min(2 ** attempt, 30)) + random.uniform(0, 1)
            kind = 'throttle_429' if last.status_code == 429 else 'server_5xx'
            _record_throttle(kind, url, method, last.status_code, attempt + 1, wait)
            time.sleep(wait); continue
        if last.status_code in RETRY_STATUS:  # final attempt still rate-limited/5xx
            _record_throttle('retry_exhausted', url, method, last.status_code, attempt + 1, 0)
        return last
    return last


## Endpoint pools (round-robin across N Doc Intelligence + Azure OpenAI resources)
To lift the Doc Intelligence pages/min and Azure OpenAI embeddings TPM ceilings at scale, both
services are read from **comma-separated pools** (`doc_intelligence_endpoints` /
`aoai_endpoints`), each falling back to the singular `*_endpoint` key so a single-resource setup
still works unchanged. Every request picks the next endpoint via a **thread-safe round-robin**
(safe under the `ThreadPoolExecutor`). Keyless Entra auth uses the same
`cognitiveservices.azure.com` token audience for *every* endpoint, so one token spans the whole
pool — the running identity just needs the Cognitive Services role on each resource. This scales
the effective quota ~N× (N endpoints) instead of packing more work into each call.


In [ ]:
class _RR:
    """Thread-safe round-robin over a list of endpoints."""
    def __init__(self, items):
        self._items = list(items); self._i = 0; self._lock = threading.Lock()
    def next(self):
        with self._lock:
            v = self._items[self._i % len(self._items)]; self._i += 1; return v
    def __len__(self):
        return len(self._items)

def _pool(plural_key, singular_key):
    raw = cfg.get(plural_key) or cfg.get(singular_key) or ''
    eps = [e.strip().rstrip('/') for e in raw.split(',') if e.strip()]
    return _RR(eps or [''])

DI_POOL = _pool('doc_intelligence_endpoints', 'doc_intelligence_endpoint')
AOAI_POOL = _pool('aoai_endpoints', 'aoai_endpoint')
print(f'DI endpoints in pool: {len(DI_POOL)} | AOAI endpoints in pool: {len(AOAI_POOL)}')


## ACL resolution (nearest-ancestor, efficient for deep paths)
Build a sorted prefix map once; each file resolves by walking up its path prefixes.


In [ ]:
acls_path = cfg.get('acls_file_path', 'Files/acls/acls.json')
try:
    raw = spark.read.text(acls_path, wholetext=True).collect()[0][0]
    ACL_MAP = {f['path'].rstrip('/'): f['groups'] for f in json.loads(raw).get('folders', [])}
except Exception as e:
    print('WARNING: no ACL file ->', e); ACL_MAP = {}

def resolve_groups(rel_path):
    """Return (groups, acl_version) using the nearest ancestor folder in ACL_MAP."""
    parts = rel_path.split('/')
    for i in range(len(parts) - 1, 0, -1):
        prefix = '/'.join(parts[:i])
        if prefix in ACL_MAP:
            groups = sorted(ACL_MAP[prefix])
            ver = hashlib.sha256(('|'.join(groups)).encode()).hexdigest()[:16]
            return groups, ver
    return [], None


## Path / folder helper
`file_path` is the **src_key** (bucket-relative). `folder_path` (parent dir) is stamped on each
index doc for folder-based filtering/faceting.


In [ ]:
def folder_of(src_key):
    return src_key.rsplit('/', 1)[0] if '/' in src_key else ''


## Extract: text files direct, everything else via Doc Intelligence (REST)
**Text-native files** (`.txt`, `.md`) are read directly — Document Intelligence rejects
`text/plain` with a 400. **All other supported types** go through DI `prebuilt-layout` over REST
with an Entra `Bearer` token; we poll `Operation-Location` (bounded by a deadline) then read text.
DI puts PDF text in `pages[].lines[]` but Office (`.docx`/`.pptx`/`.xlsx`) text in `paragraphs`
/`content`, so we fall back through all three so nothing extracts empty.


In [ ]:
DI_API_VERSION = '2024-11-30'
TEXT_EXTS = {'txt', 'md'}          # read directly; DI 400s on text/plain
DI_POLL_DEADLINE_S = 180           # hard cap so a file can never hang the run

class SourceMissing(Exception):
    """The file was listed by nb_01 but is no longer present at the source (deleted from S3)."""
    pass

def read_shortcut_bytes(rel, local):
    """Read a file's bytes from the OneLake shortcut robustly. S3-shortcut content is not always
    faulted into the local POSIX mount (/lakehouse/default/Files/...), which surfaces as
    FileNotFoundError even though nb_01 listed the file. Strategy: retry the fast local read
    (handles transient fault-in lag), then fall back to the Fabric fs API (resolves shortcuts)
    copying to a temp path. If every method reports not-found (the file was deleted from S3 after
    nb_01 scanned it), raise SourceMissing so the caller can mark it terminally."""
    last = None
    for attempt in range(3):
        try:
            with open(local, 'rb') as f:
                return f.read()
        except FileNotFoundError as e:
            last = e
            try:
                import notebookutils, uuid as _uuid
                tmp = f'/tmp/{_uuid.uuid4().hex}_{os.path.basename(local)}'
                notebookutils.fs.cp(rel, f'file://{tmp}')
                with open(tmp, 'rb') as f:
                    data = f.read()
                try:
                    os.remove(tmp)
                except Exception:
                    pass
                return data
            except Exception as e2:
                last = e2
                time.sleep(2 ** attempt)
    msg = str(last)
    if any(s in msg for s in ('Not Found', '404', 'PATH_NOT_FOUND', 'No such file')):
        raise SourceMissing(f'source no longer present in shortcut: {rel}')
    raise last

def _read_text_file(src_key):
    return [(1, src_read_bytes(src_key).decode('utf-8', errors='replace'))]

def _di_pages(analyze_result):
    """Build [(page_number, text)] resiliently across PDF + Office shapes."""
    page_text = {}
    # 1) PDFs/images: lines per page
    for p in analyze_result.get('pages', []):
        lines = [l.get('content', '') for l in p.get('lines', [])]
        if lines:
            page_text[p.get('pageNumber', 1)] = '\n'.join(lines)
    # 2) Office docs: text lives in paragraphs (tagged with a page in boundingRegions)
    if not page_text:
        for para in analyze_result.get('paragraphs', []):
            pn = (para.get('boundingRegions') or [{}])[0].get('pageNumber', 1)
            prev = page_text.get(pn, '')
            page_text[pn] = (prev + '\n' + para.get('content', '')) if prev else para.get('content', '')
    # 3) last resort: the whole concatenated content on page 1
    if not page_text:
        content = analyze_result.get('content', '')
        if content.strip():
            page_text[1] = content
    return sorted(page_text.items())

def extract_pages(src_key, ext):
    """Return list of (page_number, text). Raises on DI failure."""
    if ext in TEXT_EXTS:
        return _read_text_file(src_key)
    data = src_read_bytes(src_key)
    ep = DI_POOL.next()
    model = cfg['doc_intelligence_model']
    url = f'{ep}/documentintelligence/documentModels/{model}:analyze?api-version={DI_API_VERSION}'
    headers = {'Authorization': f'Bearer {cog_token()}', 'Content-Type': 'application/octet-stream'}
    r = http('POST', url, headers=headers, data=data)
    r.raise_for_status()
    op_url = r.headers['Operation-Location']
    deadline = time.time() + DI_POLL_DEADLINE_S
    while True:
        time.sleep(2)
        pr = http('GET', op_url, headers={'Authorization': f'Bearer {cog_token()}'})
        pr.raise_for_status()
        j = pr.json()
        status = j.get('status')
        if status in ('succeeded', 'failed', 'completed'):
            break
        if time.time() > deadline:
            raise TimeoutError(f'DI poll exceeded {DI_POLL_DEADLINE_S}s (status={status})')
    if j.get('status') == 'failed':
        raise RuntimeError(f'DI analyze failed: {j}')
    return _di_pages(j['analyzeResult'])

def chunk_pages(pages):
    """Chunk by page (no cross-chunk overlap), matching AI Search's default page chunking.
    Each page becomes one chunk tagged with its page number. A page longer than CHUNK_SIZE
    is split into multiple non-overlapping chunks that keep the same page number (safety cap
    so a huge page doesn't exceed embedding limits)."""
    out = []
    for page_number, text in pages:
        text = (text or '').strip()
        if not text:
            continue
        if len(text) <= CHUNK_SIZE:
            out.append((page_number, text))
        else:
            for i in range(0, len(text), CHUNK_SIZE):
                piece = text[i:i + CHUNK_SIZE].strip()
                if piece:
                    out.append((page_number, piece))
    return out


## Source read accessor (mode-aware)
`src_read_bytes(src_key)` reads from the OneLake shortcut (local mount) or directly from S3
(SigV4 REST) per `source_mode`. Used by `extract_pages` above.


In [ ]:
def src_to_shortcut_rel(src_key):
    """src_key (bucket-relative) -> (lakehouse rel path, local POSIX mount path) for shortcut mode."""
    rel = SHORTCUT_ROOT + "/" + src_key
    return rel, "/lakehouse/default/" + rel

def src_read_bytes(src_key):
    """Return the object bytes for src_key, honoring SOURCE_MODE. Raises SourceMissing if gone."""
    if SOURCE_MODE == "s3_direct":
        try:
            return s3_get_bytes(S3_ENDPOINT, S3_BUCKET, src_key, S3_REGION, _S3_AK, _S3_SK,
                                addressing=S3_ADDRESSING, verify=S3_VERIFY_TLS)
        except requests.HTTPError as e:
            code = getattr(getattr(e, "response", None), "status_code", None)
            if code in (403, 404):
                raise SourceMissing("s3 object not present: " + src_key)
            raise
    rel, local = src_to_shortcut_rel(src_key)
    return read_shortcut_bytes(rel, local)


## Embed (Azure OpenAI REST)
Embeddings via `POST /openai/deployments/{deployment}/embeddings` with an Entra `Bearer` token.
`input` accepts the whole chunk list in one call; the response preserves order.


In [ ]:
def embed(texts):
    ep = AOAI_POOL.next()
    url = f'{ep}/openai/deployments/{EMB_DEPLOY}/embeddings?api-version={AOAI_API}'
    headers = {'Authorization': f'Bearer {cog_token()}', 'Content-Type': 'application/json'}
    r = http('POST', url, headers=headers, json_body={'input': texts})
    r.raise_for_status()
    return [d['embedding'] for d in r.json()['data']]

def chunk_id(file_path, idx):
    h = hashlib.sha256(file_path.encode()).hexdigest()[:32]
    return f'{h}-{idx}'

def build_docs(file_path, file_name, ext, chunks, vectors, groups, indexed_utc,
               folder_path='', file_size=None, last_modified=None):
    docs = []
    for idx, ((page, text), vec) in enumerate(zip(chunks, vectors)):
        docs.append({
            'chunk_id': chunk_id(file_path, idx),
            'file_path': file_path, 'file_name': file_name, 'file_extension': ext,
            'content': text, 'content_vector': vec,
            'page_number': int(page), 'chunk_index': idx,
            'folder_path': folder_path,
            'file_size': (int(file_size) if file_size is not None else None),
            'last_modified': last_modified,
            'author': None, 'last_modified_by': None,  # reserved (not available from S3 listing)
            'allowed_groups': groups,
            'embedding_model': EMB_DEPLOY, 'chunk_strategy_version': STRATEGY,
            'indexed_utc': indexed_utc,
        })
    return docs


## Search write helpers (REST: query ids, delete-by-file, batched upload)
All via the AI Search REST API with the `api-key` header. `delete_file_chunks` removes any
existing chunks for a file (deterministic ids) so re-ingest never duplicates.


In [ ]:
def _search_headers():
    return {'api-key': SEARCH_KEY, 'Content-Type': 'application/json'}

def delete_file_chunks(file_path):
    """Delete all existing chunks for a file. Pages through results in case of many chunks."""
    url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={SEARCH_API}'
    safe = file_path.replace("'", "''")
    ids, skip = [], 0
    while True:
        body = {'search': '*', 'filter': f"file_path eq '{safe}'",
                'select': 'chunk_id', 'top': 1000, 'skip': skip}
        r = http('POST', url, headers=_search_headers(), json_body=body)
        r.raise_for_status()
        page = r.json().get('value', [])
        ids += [d['chunk_id'] for d in page]
        if len(page) < 1000:
            break
        skip += 1000
    if ids:
        _index_batch([{'@search.action': 'delete', 'chunk_id': i} for i in ids])
    return len(ids)

def _index_batch(actions):
    url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/index?api-version={SEARCH_API}'
    for i in range(0, len(actions), 500):
        r = http('POST', url, headers=_search_headers(), json_body={'value': actions[i:i + 500]})
        r.raise_for_status()

def upload_docs(docs):
    _index_batch([{**d, '@search.action': 'mergeOrUpload'} for d in docs])


## Status helpers


In [ ]:
def set_status(file_path, status, reason=None, inc_retry=False):
    now = datetime.now(timezone.utc)
    sets = {'process_status': F.lit(status), 'status_reason': F.lit(reason),
            'status_updated_utc': F.lit(now)}
    if inc_retry:
        sets['retry_count'] = F.coalesce(F.col('retry_count'), F.lit(0)) + 1
    (DeltaTable.forName(spark, 'file_metadata').update(
        condition=F.col('file_path') == F.lit(file_path), set=sets))

def log_success(file_path, chunks, pages, duration_ms):
    row = [(file_path, chunks, pages, duration_ms, EMB_DEPLOY, cfg['doc_intelligence_model'],
            RUN_ID, datetime.now(timezone.utc))]
    spark.createDataFrame(row, 'file_path string, chunks int, pages int, duration_ms long, embedding_model string, di_model string, run_id string, ts_utc timestamp')\
        .write.mode('append').saveAsTable('ingestion_log')

def log_skip(file_path, reason, detail):
    row = [(file_path, reason, str(detail)[:4000], RUN_ID, datetime.now(timezone.utc))]
    spark.createDataFrame(row, 'file_path string, reason string, detail string, run_id string, ts_utc timestamp')\
        .write.mode('append').saveAsTable('skipped_log')

def upsert_state(file_path, change_hash, acl_version, chunk_count):
    now = datetime.now(timezone.utc)
    src = spark.createDataFrame(
        [(file_path, change_hash, acl_version, chunk_count, EMB_DEPLOY, STRATEGY, now)],
        'file_path string, change_hash string, acl_version string, chunk_count int, embedding_model string, chunk_strategy_version string, indexed_utc timestamp')
    (DeltaTable.forName(spark, 'ingestion_state').alias('t')
       .merge(src.alias('s'), 't.file_path = s.file_path')
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


## Reclaim stuck files, then claim the work queue
The `pending`→`ingesting` transition is the claim; a run only processes what it claims.

**Visibility-timeout / lease reclaim:** a file whose `status_updated_utc` has sat in `ingesting`
longer than `ingesting_lease_minutes` means a prior run crashed, was cancelled, or the file is a
**poison pill** that hangs every attempt. Each reclaim **counts as a retry** (`retry_count += 1`):
under `max_retries` it goes back to the queue as `reingest`; at/over the cap it is **dead-lettered**
so one stuck file can never loop forever. This is what makes the pipeline safe to **stop and
restart** — a killed run leaves no permanently-stranded work.


In [ ]:
WORK_STATUSES = ['new', 'changed', 'reingest', 'error']
LEASE_MIN = int(cfg.get('ingesting_lease_minutes', '120'))

from datetime import timedelta
now = datetime.now(timezone.utc)
cutoff = now - timedelta(minutes=LEASE_MIN)
fm = DeltaTable.forName(spark, 'file_metadata')
stale = ((F.col('process_status') == 'ingesting') &
         (F.coalesce(F.col('status_updated_utc'), F.lit(datetime(1970, 1, 1))) < F.lit(cutoff)))
attempt_now = F.coalesce(F.col('retry_count'), F.lit(0)) + F.lit(1)
over_cap = stale & (attempt_now >= F.lit(MAX_RETRIES))
under_cap = stale & (attempt_now < F.lit(MAX_RETRIES))
n_dead = fm.toDF().where(over_cap).count()
n_retry = fm.toDF().where(under_cap).count()
if n_dead:
    fm.update(condition=over_cap,
        set={'process_status': F.lit('dead_letter'),
             'status_reason': F.lit('stuck_ingesting_exceeded_retries'),
             'retry_count': attempt_now,
             'status_updated_utc': F.lit(now)})
if n_retry:
    fm.update(condition=under_cap,
        set={'process_status': F.lit('reingest'),
             'status_reason': F.lit('recovered_stale_ingesting'),
             'retry_count': attempt_now,
             'status_updated_utc': F.lit(now)})
if n_dead or n_retry:
    print(f'stale ingesting reclaimed: reingest={n_retry} dead_letter={n_dead} (lease={LEASE_MIN}m)')

# Handle deletions first: purge chunks, drop state, mark done.
deleted = [r['file_path'] for r in spark.table('file_metadata')
           .where(F.col('process_status') == 'deleted').select('file_path').collect()]
if deleted:
    for fp in deleted:
        try:
            n = delete_file_chunks(fp)
            safe = fp.replace("'", "''")
            spark.sql(f"DELETE FROM ingestion_state WHERE file_path = '{safe}'")
            (DeltaTable.forName(spark, 'file_metadata').update(
                condition=F.col('file_path') == F.lit(fp),
                set={'process_status': F.lit('complete'),
                     'status_reason': F.lit('deleted_purged'),
                     'status_updated_utc': F.lit(datetime.now(timezone.utc))}))
            print(f'purged {n} chunks for deleted file {fp}')
        except Exception as e:
            log_skip(fp, 'delete_error', e)

# NOTE: work is NOT bulk-claimed here. The drain loop below claims one small batch at a time
# and marks ONLY those files 'ingesting' right before processing them — so the status always
# reflects what is actually being processed, and a stopped run leaves at most one batch in flight.


## Process the queue in batches (drain loop)
Rather than one giant run, the job **loops over small batches** until the queue is drained (or
a `max_batches_per_run` / `run_time_budget_min` cap is hit). Each iteration:

1. **Claims exactly one batch** (`batch_size` files) and marks **only those** `ingesting` — so
   `process_status` always reflects what is *actually* being processed. A stopped/crashed run
   leaves at most one batch in flight, which the stale-lease reclaim recovers.
2. **Processes** the batch: network-heavy work (DI → embed → Search) fans out in a bounded
   `ThreadPoolExecutor(max_concurrency)` across the round-robin endpoint pools; Delta
   status/state/log writes are then applied in a **single commit per batch** on the driver
   (avoids optimistic-concurrency conflicts *and* the per-file whole-table rewrite that
   dominates cost at scale). Any rate-limit/retry events are flushed to `throttle_log`.
3. Updates `run_progress`, then asks **"any more?"** and repeats.

The claim is scale-safe and self-excluding: it only picks rows last touched **before this run
started** (`RUN_START_TS`), so a file that errors mid-run is retried on the *next* invocation —
never hot-looped within this one.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Live progress via a `run_progress` DELTA table (single source of truth) ----------------
# One row per run, updated in place (MERGE on run_id). Delta uses optimistic-concurrency / MVCC,
# so there are NO file locks: nb_05_status and SQL can read live progress while the run writes,
# and readers never block the writer. The write is THROTTLED (>= every DELTA_EVERY_S, plus on
# phase change / completion) so a per-file cadence never bloats the commit log or contends with
# the file_metadata status writes. A progress write never fails the run.
RUN_T0 = time.time()
DELTA_EVERY_S = 15
_last_delta = [0.0]

spark.sql('''CREATE TABLE IF NOT EXISTS run_progress (
    run_id string, phase string, processed int, total_claimed int,
    complete int, error int, skipped int, last_file string, last_outcome string,
    elapsed_s double, rate_per_min double, eta_min double, updated_utc timestamp
) USING delta''')

# Explicit schema so single-row writes with None fields (e.g. eta_min / last_file at the
# 'started' phase) become TYPED nulls — otherwise Spark infers NullType and the MERGE fails.
_RP_SCHEMA = T.StructType([
    T.StructField('run_id', T.StringType()), T.StructField('phase', T.StringType()),
    T.StructField('processed', T.IntegerType()), T.StructField('total_claimed', T.IntegerType()),
    T.StructField('complete', T.IntegerType()), T.StructField('error', T.IntegerType()),
    T.StructField('skipped', T.IntegerType()), T.StructField('last_file', T.StringType()),
    T.StructField('last_outcome', T.StringType()), T.StructField('elapsed_s', T.DoubleType()),
    T.StructField('rate_per_min', T.DoubleType()), T.StructField('eta_min', T.DoubleType()),
    T.StructField('updated_utc', T.TimestampType())])

# --- throttle_log: durable record of rate-limiting/retries, flushed once per batch -----------
spark.sql('''CREATE TABLE IF NOT EXISTS throttle_log (
    run_id string, ts_utc timestamp, host string, method string,
    status int, attempt int, wait_s double, kind string
) USING delta''')
_THROTTLE_SCHEMA = T.StructType([
    T.StructField('run_id', T.StringType()), T.StructField('ts_utc', T.TimestampType()),
    T.StructField('host', T.StringType()), T.StructField('method', T.StringType()),
    T.StructField('status', T.IntegerType()), T.StructField('attempt', T.IntegerType()),
    T.StructField('wait_s', T.DoubleType()), T.StructField('kind', T.StringType())])

def flush_throttle():
    """Append any buffered throttle/retry events to throttle_log (one commit per batch). A
    throttle-log write never fails the run."""
    with _THROTTLE_LOCK:
        if not _THROTTLE_BUF:
            return 0
        batch = list(_THROTTLE_BUF); _THROTTLE_BUF.clear()
    rows = [(RUN_ID, ts, host, method, status, attempt, wait_s, kind)
            for (ts, host, method, status, attempt, wait_s, kind) in batch]
    try:
        (spark.createDataFrame(rows, schema=_THROTTLE_SCHEMA)
            .write.mode('append').saveAsTable('throttle_log'))
    except Exception as _e:
        print('throttle_log write failed (non-fatal):', _e)
    return len(rows)

def _progress_row(phase, processed, total, tallies, last_file, last_outcome):
    elapsed = time.time() - RUN_T0
    rate = (processed / elapsed * 60) if elapsed > 0 and processed else 0.0
    eta = round(max(total - processed, 0) / rate, 1) if rate > 0 else None
    return {'run_id': RUN_ID, 'phase': phase, 'processed': int(processed),
            'total_claimed': int(total), 'complete': int(tallies.get('complete', 0)),
            'error': int(tallies.get('error', 0)), 'skipped': int(tallies.get('skipped', 0)),
            'last_file': last_file, 'last_outcome': last_outcome,
            'elapsed_s': round(elapsed, 1), 'rate_per_min': round(rate, 1), 'eta_min': eta}

def _write_progress_delta(row):
    now_ts = datetime.now(timezone.utc)
    ordered = {f.name: (now_ts if f.name == 'updated_utc' else row.get(f.name)) for f in _RP_SCHEMA}
    src = spark.createDataFrame([ordered], schema=_RP_SCHEMA)
    (DeltaTable.forName(spark, 'run_progress').alias('t')
       .merge(src.alias('s'), 't.run_id = s.run_id')
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

def hb(phase, processed, total, tallies, last_file=None, last_outcome=None):
    try:  # throttled Delta upsert (always on phase change / completion)
        if phase in ('started', 'done') or (time.time() - _last_delta[0]) >= DELTA_EVERY_S:
            _write_progress_delta(_progress_row(phase, processed, total, tallies, last_file, last_outcome))
            _last_delta[0] = time.time()
    except Exception as _e:
        print('run_progress delta write failed (non-fatal):', _e)

def compute(w):
    """Pure network/compute; NO Delta writes. Returns a result dict."""
    fp = w['file_path']; ext = (w['file_extension'] or '').lower()  # fp == src_key
    groups, acl_version = resolve_groups(fp)
    if not groups and not ACL_BYPASS:
        return {'fp': fp, 'status': 'skipped', 'reason': 'no_acl', 'detail': fp}
    if ext not in SUPPORTED:
        return {'fp': fp, 'status': 'skipped', 'reason': 'doc_intel_unsupported', 'detail': ext}
    t0 = time.time()
    stage = 'extract'
    try:
        pages = extract_pages(fp, ext)
        chunks = chunk_pages(pages)
        if not chunks:
            return {'fp': fp, 'status': 'skipped', 'reason': 'empty_extract', 'detail': fp}
        stage = 'embed'
        vectors = embed([c[1] for c in chunks])
        stage = 'search'
        delete_file_chunks(fp)  # re-ingest cleanup
        now_iso = datetime.now(timezone.utc).isoformat()
        lm = w.get('modified_datetime')
        lm_iso = lm.isoformat() if hasattr(lm, 'isoformat') else (str(lm) if lm else None)
        docs = build_docs(fp, w['file_name'], ext, chunks, vectors, groups, now_iso,
                          folder_path=folder_of(fp), file_size=w.get('file_size'),
                          last_modified=lm_iso)
        upload_docs(docs)
        return {'fp': fp, 'status': 'complete', 'change_hash': w['change_hash'],
                'acl_version': acl_version, 'chunks': len(docs), 'pages': len(pages),
                'duration_ms': int((time.time() - t0) * 1000)}
    except SourceMissing as e:
        # Deleted from S3 after nb_01 scanned it — terminal, not a retryable error. nb_01's
        # deletion sweep will formally tombstone it (and purge any chunks) on its next run.
        return {'fp': fp, 'status': 'skipped', 'reason': 'source_missing', 'detail': str(e)}
    except Exception as e:
        # Capture the failing STAGE, a concise message, and the full traceback for the log.
        return {'fp': fp, 'status': 'error', 'stage': stage,
                'error': f'{type(e).__name__}: {e}', 'trace': traceback.format_exc()}

# Explicit schemas so batched single-row-per-file writes never infer NullType (e.g. an all-
# complete batch has status_reason=None for every row).
_FM_SCHEMA = T.StructType([
    T.StructField('file_path', T.StringType()), T.StructField('process_status', T.StringType()),
    T.StructField('status_reason', T.StringType()), T.StructField('retry_count', T.IntegerType()),
    T.StructField('status_updated_utc', T.TimestampType())])
_STATE_SCHEMA = T.StructType([
    T.StructField('file_path', T.StringType()), T.StructField('change_hash', T.StringType()),
    T.StructField('acl_version', T.StringType()), T.StructField('chunk_count', T.IntegerType()),
    T.StructField('embedding_model', T.StringType()),
    T.StructField('chunk_strategy_version', T.StringType()),
    T.StructField('indexed_utc', T.TimestampType())])
_LOG_SCHEMA = T.StructType([
    T.StructField('file_path', T.StringType()), T.StructField('chunks', T.IntegerType()),
    T.StructField('pages', T.IntegerType()), T.StructField('duration_ms', T.LongType()),
    T.StructField('embedding_model', T.StringType()), T.StructField('di_model', T.StringType()),
    T.StructField('run_id', T.StringType()), T.StructField('ts_utc', T.TimestampType())])
_SKIP_SCHEMA = T.StructType([
    T.StructField('file_path', T.StringType()), T.StructField('reason', T.StringType()),
    T.StructField('detail', T.StringType()), T.StructField('run_id', T.StringType()),
    T.StructField('ts_utc', T.TimestampType())])

def apply_batch(results):
    """Commit a whole batch's outcomes in ONE set of Delta writes instead of per file: a single
    MERGE into file_metadata (status/reason/retry_count), a single MERGE into ingestion_state,
    and one append each to ingestion_log / skipped_log. On an unpartitioned table each .update()
    rewrites the whole table, so per-file commits are O(N^2) over a run; per-batch commits make
    the status writes keep up at 100k-file scale. Returns per-outcome tallies for this batch."""
    tally = {'complete': 0, 'skipped': 0, 'error': 0}
    if not results:
        return tally
    now = datetime.now(timezone.utc)
    paths = [r['fp'] for r in results]
    rc_map = {row['file_path']: (row['retry_count'] or 0)
              for row in spark.table('file_metadata').where(F.col('file_path').isin(paths))
                   .select('file_path', 'retry_count').collect()}
    fm_rows, state_rows, log_rows, skip_rows = [], [], [], []
    for r in results:
        fp = r['fp']; cur = rc_map.get(fp, 0)
        if r['status'] == 'skipped':
            tally['skipped'] += 1
            fm_rows.append((fp, 'skipped', r['reason'], cur, now))
            skip_rows.append((fp, r['reason'], str(r.get('detail'))[:4000], RUN_ID, now))
        elif r['status'] == 'complete':
            tally['complete'] += 1
            fm_rows.append((fp, 'complete', None, cur, now))
            state_rows.append((fp, r['change_hash'], r['acl_version'], r['chunks'],
                               EMB_DEPLOY, STRATEGY, now))
            log_rows.append((fp, r['chunks'], r['pages'], r['duration_ms'],
                             EMB_DEPLOY, cfg['doc_intelligence_model'], RUN_ID, now))
        else:  # error -> retry, or dead_letter once the retry cap is hit
            tally['error'] += 1
            rc = cur + 1
            reason = f"{r.get('stage', 'ingest')}_error"
            msg = r['error'][:1000]
            trace = str(r.get('trace', r['error']))[:4000]
            if rc >= MAX_RETRIES:
                fm_rows.append((fp, 'dead_letter', msg, rc, now))
                skip_rows.append((fp, 'dead_letter', trace, RUN_ID, now))
            else:
                fm_rows.append((fp, 'error', msg, rc, now))
                skip_rows.append((fp, reason, trace, RUN_ID, now))
    # 1) file_metadata status transition (single MERGE for the whole batch)
    src = spark.createDataFrame(fm_rows, schema=_FM_SCHEMA)
    (DeltaTable.forName(spark, 'file_metadata').alias('t')
       .merge(src.alias('s'), 't.file_path = s.file_path')
       .whenMatchedUpdate(set={'process_status': 's.process_status',
                               'status_reason': 's.status_reason',
                               'retry_count': 's.retry_count',
                               'status_updated_utc': 's.status_updated_utc'})
       .execute())
    # 2) ingestion_state for completed files (single MERGE)
    if state_rows:
        st = spark.createDataFrame(state_rows, schema=_STATE_SCHEMA)
        (DeltaTable.forName(spark, 'ingestion_state').alias('t')
           .merge(st.alias('s'), 't.file_path = s.file_path')
           .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    # 3) append-only logs (one commit each)
    if log_rows:
        (spark.createDataFrame(log_rows, schema=_LOG_SCHEMA)
            .write.mode('append').saveAsTable('ingestion_log'))
    if skip_rows:
        (spark.createDataFrame(skip_rows, schema=_SKIP_SCHEMA)
            .write.mode('append').saveAsTable('skipped_log'))
    return tally

results = {}
RUN_START_TS = datetime.now(timezone.utc)

def _work_query():
    # Rows still needing work, untouched since this run began (so files we process/err this
    # run are not re-claimed until the NEXT invocation). Pushed entirely into Spark = scale-safe.
    return (spark.table('file_metadata')
            .where(F.col('process_status').isin(WORK_STATUSES))
            .where(F.coalesce(F.col('retry_count'), F.lit(0)) < MAX_RETRIES)
            .where(F.coalesce(F.col('status_updated_utc'), F.lit(datetime(1970,1,1))) < F.lit(RUN_START_TS)))

def claim_batch(n):
    batch = [r.asDict() for r in _work_query()
             .select('file_path','file_name','file_extension','change_hash',
                     'file_size','modified_datetime')
             .orderBy('status_updated_utc','file_path').limit(n).collect()]
    if batch:            # mark ONLY this batch 'ingesting', in a SINGLE commit (not per-file)
        paths = [w['file_path'] for w in batch]
        (DeltaTable.forName(spark, 'file_metadata').update(
            condition=F.col('file_path').isin(paths),
            set={'process_status': F.lit('ingesting'),
                 'status_updated_utc': F.lit(datetime.now(timezone.utc))}))
    return batch

def process_batch(batch, tallies, done, total):
    """Fan out the batch's network work in the thread pool (live per-file heartbeat), then
    commit all outcomes in a single per-batch apply_batch, and flush this batch's throttle
    events. Delta writes never overlap the pool, avoiding OCC conflicts."""
    results = []
    with ThreadPoolExecutor(max_workers=MAX_CONC) as pool:
        futures = [pool.submit(compute, w) for w in batch]
        for fut in as_completed(futures):
            r = fut.result()
            results.append(r)
            outcome = r['status']            # 'complete' | 'skipped' | 'error'
            tallies[outcome] = tallies.get(outcome, 0) + 1
            done += 1
            hb('running', done, total, tallies, last_file=r.get('fp'), last_outcome=outcome)
            print(f"  [{done}/{total}] {outcome:8s} {r.get('fp')}")
    apply_batch(results)     # single-commit-per-batch status/state/log writes
    n_thr = flush_throttle() # persist any 429/5xx/network retry events from this batch
    if n_thr:
        print(f'  (throttle: logged {n_thr} rate-limit/retry event(s) this batch)')
    return done

backlog = _work_query().count()   # initial queue depth, for progress + ETA
print(f'queue depth at start: {backlog}  (batch_size={BATCH}, max_batches={MAX_BATCHES or "drain"})')
hb('started', 0, backlog, results)
done = 0; batch_no = 0
while True:
    if MAX_BATCHES and batch_no >= MAX_BATCHES:
        print(f'stop: reached max_batches_per_run={MAX_BATCHES}'); break
    if RUN_BUDGET_S and (time.time() - RUN_T0) > RUN_BUDGET_S:
        print(f'stop: reached run_time_budget_min ({RUN_BUDGET_S//60}m)'); break
    batch = claim_batch(BATCH)
    if not batch:
        print('queue drained — no more work to claim'); break
    batch_no += 1
    print(f'--- batch {batch_no}: claimed {len(batch)} file(s), marked ingesting ---')
    done = process_batch(batch, results, done, backlog)
hb('done', done, backlog, results)
print(f'run complete after {batch_no} batch(es):', results)


## Summary


In [ ]:
spark.table('file_metadata').groupBy('process_status').count().orderBy('process_status').show()
print('--- skipped this run ---')
spark.table('skipped_log').where(F.col('run_id') == RUN_ID).show(truncate=False)
